In [255]:
import numpy as np

In [256]:
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]


In [257]:
labels = [row[3] for row in data]

In [258]:
labels

['Wine', 'Beer', 'Whiskey', 'Wine', 'Beer', 'Whiskey', 'Wine', 'Beer']

In [259]:
one_hot_labels = {'Wine':0,'Beer':1,'Whiskey':2}

In [260]:
labels = [one_hot_labels[label] for label in labels]

In [261]:
labels

[0, 1, 2, 0, 1, 2, 0, 1]

In [262]:
y = np.array(labels)

In [263]:
features = [row[:3] for row in data]

In [264]:
features

[[12.0, 1.5, 1],
 [5.0, 2.0, 0],
 [40.0, 0.0, 1],
 [13.5, 1.2, 1],
 [4.5, 1.8, 0],
 [38.0, 0.1, 1],
 [11.5, 1.7, 1],
 [5.5, 2.3, 0]]

In [265]:
X = np.array(features)

In [266]:
def gini_impurity(z):
  z = np.array(z)
  unique, counts = np.unique(z, return_counts=True)
  p = counts/(np.shape(z)[0])
  pp = 1-p
  return 1.0 - np.sum(p ** 2)

In [267]:
float(gini_impurity(y))

0.65625

In [268]:
def weighted_gini(y_left, y_right):
    n = len(y_left) + len(y_right)
    return (len(y_left)/n) * gini_impurity(y_left) + \
           (len(y_right)/n) * gini_impurity(y_right)

In [269]:
np.shape(X)[1]

3

In [270]:
X

array([[12. ,  1.5,  1. ],
       [ 5. ,  2. ,  0. ],
       [40. ,  0. ,  1. ],
       [13.5,  1.2,  1. ],
       [ 4.5,  1.8,  0. ],
       [38. ,  0.1,  1. ],
       [11.5,  1.7,  1. ],
       [ 5.5,  2.3,  0. ]])

In [341]:
def split(Xk, yk):
  best_feature = None
  best_threshold = None
  best_gini = float('inf')
  for i in range(np.shape(Xk)[1]):
    for j in range(np.shape(Xk)[0]):
      cur_feature = Xk[:,i]
      cur_treshold = Xk[:,i][j]
      yk_left = []
      yk_right = []
      for k in range(np.shape(Xk)[0]):
        if Xk[:,i][k] <= cur_treshold:
          yk_left.append(yk[k])
        else:
          yk_right.append(yk[k])
      if len(yk_left) == 0 or len(yk_right) == 0:
        continue
      current_gini = weighted_gini(yk_left, yk_right)
      if current_gini < best_gini:
        best_gini = current_gini
        best_feature = i
        best_threshold = cur_treshold
  return best_feature, best_threshold, best_gini

In [342]:
class Node:
  def __init__(self, feature_index, threshold, left, right, value):
    self.feature_index = feature_index
    self.threshold = threshold
    self.left = left
    self.right = right
    self.value = value

In [343]:
def majority_class(yk):
  unique, count = np.unique(yk, return_counts=True)
  return unique[np.argmax(count)]

In [344]:
def is_pure(yk):
  return (len(np.unique(yk)) == 1)

In [345]:
def build_tree(Xt,yt, depth=0, max_depth=5, min_samples=2):
  if is_pure(yt) or len(yt) < min_samples or depth >= max_depth:
    return Node(None, None, None, None, majority_class(yt))
  feature, threshold, gini=split(Xt,yt)
  if feature is None or threshold is None:
    return Node(None, None, None, None, majority_class(yt))
  mask = Xt[:, feature] <= threshold
  X_left = Xt[mask]
  X_right = Xt[~mask]
  y_left = yt[mask]
  y_right = yt[~mask]
  left_child = build_tree(X_left, y_left, depth+1, max_depth, min_samples)
  right_child = build_tree(X_right, y_right, depth+1, max_depth, min_samples)

  return Node(feature_index=feature,
              threshold=threshold,
              left=left_child,
              right=right_child, value=None)

In [346]:
def predict_one(x, node):
    if node.value is not None:
      return node.value
    if node.left is None and node.right is None:  # safety guard
        return None
    if x[node.feature_index] <= node.threshold:
        return predict_one(x, node.left)
    else:
        return predict_one(x, node.right)

In [347]:
def predict(Xk, tree):
    return [predict_one(x, tree) for x in Xk]

In [348]:
test_data = np.array([
    [6., 2.1, 0],   # Expected: Beer
    [39.0, 0.05, 1], # Expected: Whiskey
    [13.0, 1.3, 1]   # Expected: Wine
])


In [349]:
X_test = np.array([row[:3] for row in test_data])

In [350]:
X_test

array([[ 6.  ,  2.1 ,  0.  ],
       [39.  ,  0.05,  1.  ],
       [13.  ,  1.3 ,  1.  ]])

In [351]:
tree = build_tree(X, y)

In [352]:
y_pred = predict(X_test, tree)

In [353]:
y_pred

[np.int64(0), np.int64(2), np.int64(0)]

In [357]:
label_names    = {v: k for k, v in one_hot_labels.items()}
print([label_names[p] for p in y_pred])

['Wine', 'Whiskey', 'Wine']


In [358]:
#I am not getting the expected output. The chosen threshold iss 5.5, but the alcohol content for beer is 6. Its classifying as wine.